# Epilepsy Diagnosis Analysis 

# ML & PREPOCESSING 

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from joblib import dump
import klib 

import warnings 
warnings.filterwarnings("ignore")

# Make plots look nice 
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('✅ All Libraries Imported Successfully!')

✅ All Libraries Imported Successfully!


# LOADING DATASET 

In [2]:
df = pd.read_csv('../data/raw_dataset/BEED_Data.csv')

In [3]:
print(f"📐 Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print('\n')
df.head()

📐 Dataset Shape: 8,000 rows x 17 columns




,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,y
0,4,7,18,25,28,27,20,10,-10,-18,-20,-16,13,32,12,10,0
1,87,114,120,106,76,54,28,5,-19,-49,-85,-102,-100,-89,-61,-21,0
2,-131,-133,-140,-131,-123,-108,-58,-51,-70,-77,-76,-76,-73,-57,-40,-14,0
3,68,104,73,34,-12,-26,-38,-36,-67,-88,-25,31,18,-4,6,-29,0
4,-67,-90,-97,-94,-86,-71,-43,-11,23,46,58,50,39,19,-9,-41,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   X1      8000 non-null   int64
 1   X2      8000 non-null   int64
 2   X3      8000 non-null   int64
 3   X4      8000 non-null   int64
 4   X5      8000 non-null   int64
 5   X6      8000 non-null   int64
 6   X7      8000 non-null   int64
 7   X8      8000 non-null   int64
 8   X9      8000 non-null   int64
 9   X10     8000 non-null   int64
 10  X11     8000 non-null   int64
 11  X12     8000 non-null   int64
 12  X13     8000 non-null   int64
 13  X14     8000 non-null   int64
 14  X15     8000 non-null   int64
 15  X16     8000 non-null   int64
 16  y       8000 non-null   int64
dtypes: int64(17)
memory usage: 1.0 MB


# SPLIT THE DATASET 

In [5]:
from src.preprocessing import DataPreprocessor 

TARGET = 'y'
df_eps = df.copy()

X = df_eps.drop('y', axis=1)
y = df_eps['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=42
)
preprocessor = DataPreprocessor()
preprocessor.find_columns(X_train)

X_train_enc, X_test_enc = preprocessor.encode(
    X_train,
    X_test
)
X_train_scaled, X_test_scaled = preprocessor.scale(
    X_train_enc,
    X_test_enc
)

In [6]:
X_train_enc.info()

<class 'pandas.DataFrame'>
Index: 6400 entries, 1467 to 7270
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   X1      6400 non-null   int64
 1   X2      6400 non-null   int64
 2   X3      6400 non-null   int64
 3   X4      6400 non-null   int64
 4   X5      6400 non-null   int64
 5   X6      6400 non-null   int64
 6   X7      6400 non-null   int64
 7   X8      6400 non-null   int64
 8   X9      6400 non-null   int64
 9   X10     6400 non-null   int64
 10  X11     6400 non-null   int64
 11  X12     6400 non-null   int64
 12  X13     6400 non-null   int64
 13  X14     6400 non-null   int64
 14  X15     6400 non-null   int64
 15  X16     6400 non-null   int64
dtypes: int64(16)
memory usage: 850.0 KB


In [7]:
X_test_enc.info()

<class 'pandas.DataFrame'>
Index: 1600 entries, 2215 to 6832
Data columns (total 16 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   X1      1600 non-null   int64
 1   X2      1600 non-null   int64
 2   X3      1600 non-null   int64
 3   X4      1600 non-null   int64
 4   X5      1600 non-null   int64
 5   X6      1600 non-null   int64
 6   X7      1600 non-null   int64
 7   X8      1600 non-null   int64
 8   X9      1600 non-null   int64
 9   X10     1600 non-null   int64
 10  X11     1600 non-null   int64
 11  X12     1600 non-null   int64
 12  X13     1600 non-null   int64
 13  X14     1600 non-null   int64
 14  X15     1600 non-null   int64
 15  X16     1600 non-null   int64
dtypes: int64(16)
memory usage: 212.5 KB


# SAVE PREPROCESSED DATASETS TO THE DATA FOLDER

In [12]:
import os 

save_dr = "/Users/murodjongafforov/Desktop/epilepsy_prediction/data/preprocessed_dataset"

os.makedirs(save_dr, exist_ok = True)

X_train_scaled.to_csv(
    os.path.join(save_dr, "X_train_scaled.csv"),
    index = True
)
X_test_scaled.to_csv(
    os.path.join(save_dr, "X_test_scaled.csv"),
    index = True
)


# ML ALGORITHMS 

# LOGISTIC REGRESSION

In [15]:
# LOGISTIC REGRESSION 
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
# Predict the class 1
y_proba = lr.predict_proba(X_test_scaled)

# EVALUATION OF LOGISTIC REGRESSION

In [33]:
class_report = classification_report(y_test, lr_pred)
conf_matrix = confusion_matrix(y_test, lr_pred)
roc_auc  = roc_auc_score(y_test, y_proba, multi_class = "ovo")

print("Classifaction Report: ")
print("\n", class_report)
print("Confusion Matrix: ")
print(conf_matrix)
print("\nRoc-Auc-Score: ")
print(f"✅ {roc_auc}")

Classifaction Report: 

               precision    recall  f1-score   support

           0       0.80      0.48      0.60       425
           1       0.48      0.59      0.53       379
           2       0.37      0.41      0.39       397
           3       0.34      0.37      0.35       399

    accuracy                           0.46      1600
   macro avg       0.49      0.46      0.47      1600
weighted avg       0.50      0.46      0.47      1600

Confusion Matrix: 
[[204  47  89  85]
 [ 12 222  83  62]
 [ 18  74 161 144]
 [ 22 123 106 148]]

Roc-Auc-Score: 
✅ 0.6566675877445197


# DECISION TREE CLASSIFIER

In [19]:
dt = DecisionTreeClassifier()
dt.fit(X_train_enc, y_train)
dt_pred = dt.predict(X_test_enc)
dt_proba = dt.predict_proba(X_test_enc)

# EVALUATION METRICS OF DECISION TREE 

In [32]:
class_report = classification_report(y_test, dt_pred)
conf_matrix = confusion_matrix(y_test, dt_pred)
roc_auc  = roc_auc_score(y_test, dt_proba, multi_class = "ovo")

print("Classifaction Report: ")
print("\n", class_report)
print("Confusion Matrix: ")
print(conf_matrix)
print("\nRoc-Auc-Score: ")
print(f"✅ {roc_auc}")

Classifaction Report: 

               precision    recall  f1-score   support

           0       0.98      0.96      0.97       425
           1       0.88      0.87      0.88       379
           2       0.84      0.87      0.85       397
           3       0.81      0.81      0.81       399

    accuracy                           0.88      1600
   macro avg       0.88      0.88      0.88      1600
weighted avg       0.88      0.88      0.88      1600

Confusion Matrix: 
[[408   6   2   9]
 [  6 331  14  28]
 [  0  13 344  40]
 [  1  25  49 324]]

Roc-Auc-Score: 
✅ 0.9186466232041625


# RANDOM FOREST CLASSIFIER

In [28]:
rf = RandomForestClassifier(
    random_state=42, 

)
rf.fit(X_train_enc, y_train)
rf_pred = rf.predict(X_test_enc)
rf_proba = rf.predict_proba(X_test_enc)

# EVALUATION METRICS OF RANDOM FOREST CLASSIFIER

In [31]:
class_report = classification_report(y_test, rf_pred)
conf_matrix = confusion_matrix(y_test, rf_pred)
roc_auc  = roc_auc_score(y_test, rf_proba, multi_class = "ovo")

print("Classifaction Report: ")
print("\n", class_report)
print("Confusion Matrix: ")
print(conf_matrix)
print("\nRoc-Auc-Score: ")
print(f"✅ {roc_auc}")

Classifaction Report: 

               precision    recall  f1-score   support

           0       1.00      1.00      1.00       425
           1       0.98      0.95      0.97       379
           2       0.93      0.95      0.94       397
           3       0.94      0.94      0.94       399

    accuracy                           0.96      1600
   macro avg       0.96      0.96      0.96      1600
weighted avg       0.96      0.96      0.96      1600

Confusion Matrix: 
[[424   0   0   1]
 [  0 361   7  11]
 [  0   6 378  13]
 [  0   2  20 377]]

Roc-Auc-Score: 
✅ 0.9972047591853888
